# Check Published Results from Public Metrics

This notebook uses de-texted public metrics derived from the original datasets. It starts with post-level brand metrics and then moves to brand-month aggregation, following the structure of the published paper.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/WindAlan-sw/luxury-brand-customer-engagement.git"
REPO_DIR = Path("luxury-brand-customer-engagement")

# In Colab, clone the repository if the public data folder is not already present.
if not Path("data/public_metrics").exists():
    try:
        if not REPO_DIR.exists():
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_DIR)
    except Exception as e:
        print("Could not clone repository. If the repo is private, upload the repository ZIP or make the repo public before using Colab.")
        print(e)

print("Working directory:", Path.cwd())
print("Public metrics folder exists:", Path("data/public_metrics").exists())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data/public_metrics")
post = pd.read_csv(DATA/"01_brand_post_metrics_detexted.csv")
month = pd.read_csv(DATA/"02_brand_month_engagement_panel.csv")
overall = pd.read_csv(DATA/"03_brand_overall_engagement_summary.csv")
sent = pd.read_csv(DATA/"09_sentiment_month_summary.csv")
cluster = pd.read_csv(DATA/"12_clustering_feature_matrix.csv")

print("Post-level metrics:", post.shape)
print("Brand-month panel:", month.shape)
display(post.head())

## Phase 1 — Post-level engagement analysis

In [ ]:
display(overall.sort_values("entropy_ce_score_mean", ascending=False))

ax = overall.sort_values("entropy_ce_score_mean").plot(x="brand", y="entropy_ce_score_mean", kind="bar", legend=False, figsize=(8,4))
ax.set_ylabel("Mean entropy CE score")
ax.set_title("Brand mean CE score from de-texted post-level metrics")
plt.tight_layout()
plt.show()

In [ ]:
score_cols = ["entropy_ce_score", "critic_ce_score", "cilos_score", "idocriw_score", "merec_score"]
post[score_cols].describe().T

## Phase 2 — Brand-month aggregation

In [ ]:
display(month.head())

monthly_posts = month.groupby("year_month")["post_count"].sum().reset_index()
ax = monthly_posts.plot(x="year_month", y="post_count", figsize=(10,4), legend=False)
ax.set_ylabel("Number of brand posts")
ax.set_title("Monthly official brand post volume")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Sentiment and clustering summaries

In [ ]:
sent_brand = sent.groupby("brand").agg(
    mean_sentiment=("sentiment_score_mean", "mean"),
    customer_posts=("customer_post_count", "sum")
).reset_index()
display(sent_brand.sort_values("mean_sentiment", ascending=False))

ax = sent_brand.sort_values("mean_sentiment").plot(x="brand", y="mean_sentiment", kind="bar", legend=False, figsize=(8,4))
ax.set_ylabel("Mean sentiment score")
ax.set_title("Aggregated sentiment by brand")
plt.tight_layout()
plt.show()

display(cluster)